# Identify change-related entity-property pairs

This notebook identifies change-related entity-property pairs in the 15,000 politician sample.

Input file: `15000politician_statements.csv.gz`

Output files:  
`15000politician_change_statements_candidates.xlsx`  
`15000politician_non_change_statements.xlsx`  
`15000politician_change_statements_final.xlsx`

The identification is conducted at the entity-property pair level. A pair is treated as change-related if it belongs to one of the following classes.

Class 1 contains directly change-expressing properties. These properties express factual change through the property-value structure itself.

Class 2 contains context-dependent properties. In the politician domain, a pair is first selected as a candidate if at least one statement satisfies one of two conditions. First, the statement contains a selected change-related qualifier. Second, the main value contains temporal or sequential information and the value type is not one of the excluded non-informative types.

The automatic candidate identification is followed by manual validation. This validation removes noisy cases, especially cases where publication date describes source metadata rather than the politician’s development history, and cases where temporal or sequential information in the main value is only part of a title, identifier, file name, or external description.

The automatic candidate identification is followed by manual validation. This validation removes candidate pairs where the selected change-related qualifier or temporal main-value signal does not refer to the development history of the politician, but instead describes such as source information, media files, names, titles, or other contextual information outside the politician’s own development.

In [ ]:
import pandas as pd
import re

input_file = "15000politician_statements.csv.gz"

doc = pd.read_csv(
    input_file,
    compression="gzip",
    dtype=str,
    keep_default_na=False
)

class1_prop_ids = {'P1317', 'P1365', 'P1366', 'P155', 'P1636', 'P2031', 'P2032', 'P4602', 'P569', 'P570', 'P580', 'P582', 'P746'}

change_qualifier_ids = {'P12506', 'P1264', 'P1319', 'P1326', 'P1365', 'P1366', 'P155', 'P156', 'P1734', 'P2937', 'P4602', 'P569', 'P570',   
    'P577', 'P580', 'P582', 'P585', 'P6949', 'P8554', 'P8555'}

Rule2_exclude_value_types = {'commonsMedia', 'external-id', 'monolingualtext', 'string', 'url', 'quantity', 'wikibase-form', 'wikibase-lexeme'}

qual_id_cols = [c for c in doc.columns if c.startswith('Qualifier_ID')]

# rule 2 for class 2
temporal_patterns = {'year':re.compile(r'\b(1[0-9]{3}|20[0-2][0-9])\b')}
sequential_patterns = {
    'ordinal':  re.compile(r'\b\d{1,3}(st|nd|rd|th)\b', re.IGNORECASE),
    'roman':    re.compile(r'\b\w+\s+'
                           r'(I|II|III|IV|V|VI|VII|VIII|IX|X|'
                           r'XI|XII|XIII|XIV|XV|XVI|XVII|XVIII|XIX|XX|'
                           r'XXI|XXII|XXIII|XXIV|XXV|XXVI|XXVII|XXVIII|XXIX|XXX)\b')
}


# get Class 1 
doc_class1 = doc[doc['Property_ID'].isin(class1_prop_ids)].copy()
doc_class1['change_source'] = 'class1'
doc_class1['change_detail'] = doc_class1['Property_ID']
print(f"Class 1 pairs:{doc_class1.groupby(['QID','Property_ID']).ngroups}")


# get Class 2  
doc_class2_candidates = doc[~doc['Property_ID'].isin(class1_prop_ids)].copy()

# annoated at statement level 
def annotate_statement(row):
    # way1: change-related qualifier
    triggered_quals = []
    for col in qual_id_cols:
        val = row[col]
        if pd.notna(val):
            qid = str(val).strip()
            if qid in change_qualifier_ids:
                triggered_quals.append(qid)
    if triggered_quals:
        return 'way1_qualifier', '|'.join(triggered_quals)

    # way2: temporal and sequential information in main value
    value_type = str(row.get('Value_Type', '')).strip()
    if value_type in Rule2_exclude_value_types:
        return '', ''

    raw_main_val = row.get('Main_Value', '')
    if pd.isna(raw_main_val):
        return '', ''
    main_val = str(raw_main_val).strip()
    if not main_val:
        return '', ''

    for pattern_name, pattern in temporal_patterns.items():
        if pattern.search(main_val):
            return 'way2_temporal', pattern_name

    for pattern_name, pattern in sequential_patterns.items():
        if pattern.search(main_val):
            return 'way2_sequential', pattern_name

    return '', ''


annotations = doc_class2_candidates.apply(annotate_statement, axis=1)
doc_class2_candidates['change_source'] = annotations.apply(lambda x: x[0])
doc_class2_candidates['change_detail'] = annotations.apply(lambda x: x[1])

# (entity, property)pair: at least one qualifierd statement
pair_has_change = doc_class2_candidates.groupby(['QID', 'Property_ID'])['change_source'].apply(
    lambda x: any(v != '' for v in x)
).reset_index()
pair_has_change.columns = ['QID', 'Property_ID', 'has_change']

c2_pairs = pair_has_change[
    pair_has_change['has_change']
][['QID', 'Property_ID']]


doc_class2 = doc_class2_candidates.merge(
    c2_pairs, on=['QID', 'Property_ID'], how='inner'
)
doc_class2['change_class'] = 'class2'
print(f"Class 2 pairs:{doc_class2.groupby(['QID','Property_ID']).ngroups}")


# merge pair of class 1 and class 2
doc_class1['change_class'] = 'class1'
doc_change = pd.concat([doc_class1, doc_class2], ignore_index=True)
doc_change = doc_change.sort_values(
    by=['QID', 'Property_ID', 'Statement_Number']
).reset_index(drop=True)

print(f"\nChange-related total: {len(doc_change)} statements")
print(f"\nChange-related pairs: {doc_change.groupby(['QID','Property_ID']).ngroups}")


# non-change
change_keys = doc_change[
    ['QID', 'Property_ID', 'Statement_Number']
].drop_duplicates()
change_keys['_in_change'] = True

doc_non_change = doc.merge(
    change_keys, on=['QID', 'Property_ID', 'Statement_Number'], how='left'
)
doc_non_change = doc_non_change[
    doc_non_change['_in_change'].isna()
].drop(columns=['_in_change'])

doc_non_change = doc_non_change.sort_values(
    by=['QID', 'Property_ID', 'Statement_Number']
).reset_index(drop=True)


print(f"Non-change-related pairs: {doc_non_change.groupby(['QID','Property_ID']).ngroups}")

doc_change.to_excel('15000politician_change_statements_candidates.xlsx', index=False)
doc_non_change.to_excel('15000politician_non_change_statements.xlsx', index=False)

print("\nSaved:")
print(" 15000politician_change_statements_candidates.xlsx")
print(" 15000politician_non_change_statements.xlsx")

Class 1 pairs:17809


## Manual validation of politician candidate pairs

The candidate extraction rules intentionally have broad recall. This is necessary because context-dependent changes may be expressed through different statement components, including qualifiers and temporally or sequentially named main values.

The candidate set is therefore manually validated. The validation applies two filters.

First, publication date (P577) is treated carefully. P577 is kept when it dates an external identifier, URL, or notable work associated with the politician. It is removed when it dates a source, media file, or external description rather than a change in the politician’s development history.

Second, candidates identified through temporal or sequential information in the main value are filtered by property. Only properties whose values can reasonably locate the politician in a temporal, event-based, or sequence context are kept.

In [2]:
import pandas as pd
df = pd.read_excel('15000politician_change_statements_candidates.xlsx')

qual_id_cols = [c for c in df.columns if c.startswith('Qualifier_ID')]

P577_keep_value_types = {'external-id', 'url'}
P577_keep_property = {'P800'}  # notable work


# Rule 2: way 2 property filtering
temporal_keep = {'P3602', 'P2715', 'P1344', 'P607', 'P793', 'P509', 'P1050', 'P463', 'P102', 'P2937', 'P39', 'P2868', 'P991'}
sequential_keep = { 'P3602', 'P463', 'P361', 'P1344', 'P39', 'P607'}


# get all qualifier IDs in one statement
def get_qualifier_ids(row):
    quals = []
    for col in qual_id_cols:
        val = row[col]
        if pd.notna(val):
            qid = str(val).strip()
            if qid:
                quals.append(qid)
    return quals

# Statement-level review flag, keep Ture
def passes_review(row):
    source = str(row.get('change_source', '')).strip()
    pid = str(row.get('Property_ID', '')).strip()
    vtype = str(row.get('Value_Type', '')).strip()

    if not source or source == 'nan':
        return None

    if source == 'class1':
        return True
        
    if source == 'way1_qualifier':
        qualifier_ids = get_qualifier_ids(row)

        if 'P577' not in qualifier_ids:
            return True

        # if P577 exists, check the value type, keep type: 'external-id', 'url'
        if vtype in P577_keep_value_types:
            return True

        # keep type is wikibase-item and proeprty is notable work
        if vtype == 'wikibase-item' and pid in P577_keep_property:
            return True

        return False

    if source == 'way2_temporal':
        return pid in temporal_keep

    if source == 'way2_sequential':
        return pid in sequential_keep

    return None

df['_passes_review'] = df.apply(passes_review, axis=1)



# Pair-level 
class1_pairs = (
    df[df['change_class'] == 'class1'][['QID', 'Property_ID']]
    .drop_duplicates()
)

class2_df = df[df['change_class'] == 'class2'].copy()

class2_pair_review = (
    class2_df
    .groupby(['QID', 'Property_ID'])['_passes_review']
    .apply(lambda values: any(v is True for v in values))
    .reset_index(name='keep_pair')
)

class2_kept_pairs = class2_pair_review[class2_pair_review['keep_pair']][['QID', 'Property_ID']]

class2_dropped_pairs = class2_pair_review[~class2_pair_review['keep_pair']][['QID', 'Property_ID']]


kept_pairs = pd.concat([class1_pairs, class2_kept_pairs],
                       ignore_index=True).drop_duplicates()

output = df.merge(
    kept_pairs,
    on=['QID', 'Property_ID'],
    how='inner'
)

output = output.drop(columns=['_passes_review'])
output = output.sort_values(by=['QID', 'Property_ID', 'Statement_Number']
                           ).reset_index(drop=True)


print(f"Class 1 pairs:{len(class1_pairs)}")
print(f"Class 2 pairs before review: {class2_df.groupby(['QID', 'Property_ID']).ngroups}")
print(f"Class 2 pairs kept:          {len(class2_kept_pairs)}")
print(f"Class 2 pairs dropped:       {len(class2_dropped_pairs)}")


print(f"Final change-related pairs: {output.groupby(['QID', 'Property_ID']).ngroups}")
output.to_excel('15000politician_change_statements_final.xlsx', index=False)
print("\nSaved: 15000politician_change_statements_final.xlsx")

Class 1 pairs:17809
Class 2 pairs before review: 16423
Class 2 pairs kept:          15412
Class 2 pairs dropped:       1011
Final change-related pairs: 33221

Saved: 15000politician_change_statements_final.xlsx
